In [27]:
import itertools
import math
import cmath

from qiskit import QuantumCircuit, QuantumRegister
from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
import ffsim
import pyscf
from pyscf import gto, scf, cc
import json
from ffsim.linalg import givens_decomposition
from qiskit_ibm_runtime import  SamplerV2 as Sampler
from qiskit.circuit.library import XXPlusYYGate, RYGate, RZZGate, CPhaseGate

from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import CouplingMap

## Molecule definition

In [28]:
def generate_hchain_geometry(natoms: int, atomic_distance: float = 0.7) -> str:
    """Returns a linear Hydrogen chain geometry for use in PySCF molecule construction.
    
    Args:
        natoms: Number of Hydrogen atoms in the chain.
        atomic_distance: Equal spacing between Hydrogen atoms.
    """
    return "; ".join([f"H 0 0 {i * atomic_distance}" for i in range(natoms)])

In [29]:
geom = generate_hchain_geometry(natoms=6)
print(geom)

mol = gto.Mole(atom = geom, charge = 0, basis = 'sto3g')
mol.build()
mf = scf.RHF(mol)
mf.verbose = 0
mf.kernel()
cc_ = cc.CCSD(mf).run()
N = mol.nao_nr() * 2

H 0 0 0.0; H 0 0 0.7; H 0 0 1.4; H 0 0 2.0999999999999996; H 0 0 2.8; H 0 0 3.5
E(CCSD) = -3.077820915918894  E_corr = -0.05767638353080283


In [30]:
# aer_sim = AerSimulator()
# pm = generate_preset_pass_manager(optimization_level = 3, backend = aer_sim)

## FFSIM LUCJ

Following https://quantum.cloud.ibm.com/docs/en/tutorials/sample-based-quantum-diagonalization.

In [31]:
# Define active space
n_frozen = 0
active_space = range(n_frozen, mol.nao_nr())


# Get molecular integrals
scf = pyscf.scf.RHF(mol).run()
norb = len(active_space)
n_electrons = int(sum(scf.mo_occ[active_space]))
n_alpha = (n_electrons + mol.spin) // 2
n_beta = (n_electrons - mol.spin) // 2
nelec = (n_alpha, n_beta)
cas = pyscf.mcscf.CASCI(scf, norb, nelec)
mo = cas.sort_mo(active_space, base=0)
hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)
eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), norb)

# Compute exact energy using FCI
reference_energy = cas.run().e_tot

print(f"norb = {norb}")
print(f"nelec = {nelec}")

converged SCF energy = -3.02014453238809
CASCI E = -3.07799545197651  E(CI) = -9.65491221626794  S^2 = 0.0000000
norb = 6
nelec = (3, 3)


In [32]:
# Get CCSD t2 amplitudes for initializing the ansatz
ccsd = pyscf.cc.CCSD(
    scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]
).run()
t1 = ccsd.t1
t2 = ccsd.t2

E(CCSD) = -3.077820915918894  E_corr = -0.05767638353080283


In [33]:
# Set ansatz properties
n_reps = 1
pairs_aa = [(p, p + 1) for p in range(norb - 1)]
pairs_ab = [(p, p) for p in range(norb)]  # None  # Let generate_lucj_pass_manager determine the alpha-beta interactions

# Initialize backend
coupling_map = CouplingMap.from_heavy_hex(3)
backend = GenericBackendV2(
    coupling_map.size(),
    coupling_map=coupling_map,
    basis_gates=["cp", "xx_plus_yy", "p", "x", "swap"],
)

# Create pass manager
pass_manager, pairs_ab = ffsim.qiskit.generate_lucj_pass_manager(
    backend=backend,
    norb=norb,
    connectivity="heavy-hex",
    interaction_pairs=(pairs_aa, pairs_ab),
    optimization_level=3,
)

# Create the LUCJ ansatz operator
ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=t2,
    t1=t1,
    n_reps=n_reps,
    interaction_pairs=(pairs_aa, pairs_ab),
    # Setting optimize=True enables the "compressed" factorization
    optimize=True,
    # Limit the number of optimization iterations to prevent the code cell from running
    # too long. Removing this line may improve results.
    options=dict(maxiter=1000),
)

# create an empty quantum circuit
qubits = QuantumRegister(2 * norb, name="q")
circuit = QuantumCircuit(qubits)

# prepare Hartree-Fock state as the reference state and append it to the quantum circuit
circuit.append(ffsim.qiskit.PrepareHartreeFockJW(norb, nelec), qubits)

# apply the UCJ operator to the reference state
circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op), qubits)
circuit.measure_all()

In [34]:
isa_circuit = pass_manager.run(circuit)
isa_circuit.draw(fold=-1)

┌──────────────────────────┐                                                                                                                                                                                                                                                                                                                                                                                                                                                               ┌─────────────────────────┐                                                                                                                                                                                                                                ┌────────────────────────┐                                                                                                                                                                                                                                                                                                                                                                            ┌──────────────────────────┐       ┌────────────┐                                                                                                                                                                  ░                                  ┌─┐
     q_11 -> 0 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤0                         ├───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────■───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤0                        ├────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤1                       ├────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤1                         ├───────┤ P(-2.7266) ├──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────░──────────────────────────────────┤M├
                    ┌───────────────────────────┐                             ┌──────────────────────────┐                                                                                │                          │┌──────────────────────────┐                                                                                                                ┌──────────────────────────┐                                                                                   │            ┌───────────────────────┐                                                                                                                                                              │                         │┌─────────────────────────┐                                                                                     ┌──────────────────────────┐                                                                                    │                        │                                                                                                               ┌───────────────────────────┐    

In [35]:
isa_circuit.count_ops()

OrderedDict([('xx_plus_yy', 48),
             ('p', 12),
             ('measure', 12),
             ('cp', 11),
             ('x', 6),
             ('swap', 2),
             ('barrier', 1)])

## Custom LUCJ

In [36]:
def orbital_rotation(qc_lucj, orbital_rotation, N):
    givens_rotations, phase_shifts = givens_decomposition(orbital_rotation)
    for c, s, i, j in givens_rotations:
        # print(i, j)
        qc_lucj.append(XXPlusYYGate(theta = 2 * math.acos(c), beta = cmath.phase(s) - 0.5 * math.pi), [i, j])
        qc_lucj.append(XXPlusYYGate(theta = 2 * math.acos(c), beta = cmath.phase(s) - 0.5 * math.pi), [i + mol.nao_nr(), j + mol.nao_nr()])

    for i, phase_shift in enumerate(phase_shifts):
        qc_lucj.p(cmath.phase(phase_shift), i)
        qc_lucj.p(cmath.phase(phase_shift), i + mol.nao_nr())

    qc_lucj.barrier()

In [37]:
def J_op(qc_lucj, diag_mat_aa, diag_mat_ab, diag_mat_bb, time, norb):
    for sigma, this_mat in enumerate([diag_mat_aa, diag_mat_bb]):
        if this_mat is None:
            print('______________________')
            print('_________NONE_________')
            print('______________________')
        if this_mat is not None:
            for i in range(norb):
                if this_mat[i, i]:
                    # print(i + sigma * norb)
                    qc_lucj.p(-0.5 * this_mat[i, i] * time, i + sigma * norb)
            for i, j in itertools.combinations(range(norb), 2):
                if this_mat[i, j]:
                    # print(i + sigma * norb, j + sigma * norb)
                    qc_lucj.cp(-this_mat[i, j] * time, i + sigma * norb, j + sigma * norb)
    # qc_lucj.barrier()
    if diag_mat_ab is not None:
        for i in range(norb):
            if diag_mat_ab[i, i]:
                qc_lucj.cp(-diag_mat_ab[i, i] * time, i, i + norb)
                
        for i, j in itertools.combinations(range(norb), 2):
            
            if diag_mat_ab[i, j]:
                qc_lucj.cp(-diag_mat_ab[i, j] * time, i, j + norb)
                
            if diag_mat_ab[j, i]:
                qc_lucj.cp(-diag_mat_ab[j, i] * time, j, i + norb)
                

    qc_lucj.barrier()

In [38]:
qc_lucj = QuantumCircuit(N)
t1, t2 = cc_.t1, cc_.t2

norb = mol.nao_nr()
pairs_aa = [(p, p + 1) for p in range(norb - 1)]   # same-spin line
pairs_ab = [(p, p) for p in range(norb)]           # opposite-spin on-site
ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=cc_.t2, t1=cc_.t1,
    n_reps=2,
    interaction_pairs=(pairs_aa, pairs_ab),
)

orbital_rotations = ucj_op.orbital_rotations
diag_mats = ucj_op.diag_coulomb_mats
final_orbital_rotation = ucj_op.final_orbital_rotation
for i in range(mol.nelectron // 2):
    qc_lucj.x(i)
    qc_lucj.x(i + mol.nao_nr())
for orb_rot, (diag_mat_aa, diag_mat_ab) in zip(orbital_rotations, diag_mats):
    orbital_rotation(qc_lucj, orb_rot.T.conj(), N)
    J_op(qc_lucj, diag_mat_aa, diag_mat_ab, diag_mat_aa, time=-1, norb=mol.nao_nr())
    orbital_rotation(qc_lucj, orb_rot, N)
if ucj_op.final_orbital_rotation is not None:
    orbital_rotation(qc_lucj, ucj_op.final_orbital_rotation, N)

In [39]:
qc_lucj_raw = QuantumCircuit(N)

for i in range(mol.nelectron // 2):
    qc_lucj_raw.x(i)
    qc_lucj_raw.x(i + mol.nao_nr())

for orb_rot, (diag_mat_aa, diag_mat_ab) in zip(orbital_rotations, diag_mats):
    orbital_rotation(qc_lucj_raw, orb_rot.T.conj(), N)
    J_op(qc_lucj_raw, diag_mat_aa, diag_mat_ab, diag_mat_aa, time=-1, norb=mol.nao_nr())
    orbital_rotation(qc_lucj_raw, orb_rot, N)

if ucj_op.final_orbital_rotation is not None:
    orbital_rotation(qc_lucj_raw, ucj_op.final_orbital_rotation, N)

qc_lucj = pass_manager.run(qc_lucj)

In [40]:
qc_lucj.draw(fold=-1)

global phase: 0
                      ┌───┐                            ┌──────────────────┐                                                                        ┌───────────────────────┐                                                                           ┌──────────────────┐                                                                                         ┌──────────────────┐                                                          ┌────────────────────────┐                                                                                              ┌─────────────────┐   ┌────────────┐                                                                     ░                                                                                                                                                                                                                                                                                                                                                                                                                           ┌──────────────────┐                                                                                                                                                    ┌──────────────────┐                                                            ┌──────────────────────────┐                                                                ┌─────────────────┐   ┌───────────┐                                                                                                                                                               ░ ┌──────────────────┐                                                                                                                 ┌───────────────────────┐                                                                     ┌──────────────────┐                                                                        ┌──────────────────┐                                                          ┌──────────────────────────┐                                                                                                      ┌─────────────────┐   ┌───────────┐                                                                    ░                                                                                                                                                                                                                                                                                                                                                                                                                  ┌──────────────────┐   ┌──────────────────┐                                                                                                                         ┌──────────────────┐                                                               ┌────────────────────────┐                                                                ┌─────────────────┐   ┌────────────┐                                                                                                                                                             ░                     ┌──────────────────┐                                                                  ┌────────────────────────┐                                                                  ┌─────────────────────────┐                                                                               ┌───────────────────────┐                                                                                        ┌─────────────────┐                                                                  ░ 
      q_1 -> 0 ───────┤ X ├────────────────────────────┤0                 ├────────────────────────────────────────────────────────────────────────┤1                      ├───────────────────────────────────────────────────────────────────────────┤0                 ├────────────────────────────────────────

In [41]:
qc_lucj.count_ops()

OrderedDict([('xx_plus_yy', 144),
             ('p', 44),
             ('swap', 42),
             ('cp', 32),
             ('barrier', 7),
             ('x', 6)])

In [42]:
# qc_lucj.measure_all()
# backend = AerSimulator()
# sampler = Sampler(mode=backend)
# pm=generate_preset_pass_manager(backend=backend,optimization_level=3)
# isa_circuit=pm.run(qc_lucj)
# #decomposed_circuit = circuit.decompose()
# pub_job = sampler.run([isa_circuit], shots=100_000)

# pub_result=pub_job.result()[0]
# print(pub_result.data.meas.get_counts())
# l1 = pub_result.data.meas.get_counts()

In [43]:
# print("orbital_rotations =",orbital_rotations)
# print("diag_mats =",diag_mats)
# print("final_orbital_rotations =",final_orbital_rotation)

In [44]:
def dump_lucj_to_json(ucj_op, orbital_rotations, diag_mats,
                      time, norb, nelectron, filepath):
    """Serialize a full LUCJ ansatz to JSON for the Julia side to consume."""

    def cx_to_pair(z):
        z = complex(z)
        return [float(z.real), float(z.imag)]

    def decompose_unitary(U):
        givens_rots, phase_shifts = givens_decomposition(U)    # <-- direct call
        rots_serialized = []
        for g in givens_rots:
            c, s, i, j = g
            rots_serialized.append({
                "c":    float(c.real) if hasattr(c, "real") else float(c),
                "s_re": float(complex(s).real),
                "s_im": float(complex(s).imag),
                "i":    int(i),
                "j":    int(j),
            })
        phases_serialized = [cx_to_pair(p) for p in phase_shifts]
        return {"givens": rots_serialized, "phase_shifts": phases_serialized}

    layers = []
    for orb_rot, (diag_mat_aa, diag_mat_ab) in zip(orbital_rotations, diag_mats):
        layers.append({
            "fwd":         decompose_unitary(orb_rot.T.conj()),
            "inv":         decompose_unitary(orb_rot),
            "diag_mat_aa": [[float(x) for x in row] for row in diag_mat_aa],
            "diag_mat_ab": [[float(x) for x in row] for row in diag_mat_ab],
        })

    final_serialized = None
    if ucj_op.final_orbital_rotation is not None:
        final_serialized = decompose_unitary(ucj_op.final_orbital_rotation)

    payload = {
        "norb":      norb,
        "time":      float(time),
        "nelectron": int(nelectron),
        "layers":    layers,
        "final":     final_serialized,
    }

    with open(filepath, "w") as f:
        json.dump(payload, f, indent=2)

    print(f"Wrote LUCJ data to {filepath}")
    print(f"  norb={norb}, time={time}, nelectron={nelectron}, "
          f"{len(layers)} layer(s), final_rotation="
          f"{'yes' if final_serialized is not None else 'no'}")

# Usage
dump_lucj_to_json(
    ucj_op, orbital_rotations, diag_mats,
    time=-1.0,
    norb=mol.nao_nr(),
    nelectron=mol.nelectron,
    filepath="lucj_params.json",
)

Wrote LUCJ data to lucj_params.json
  norb=6, time=-1.0, nelectron=6, 2 layer(s), final_rotation=yes


In [45]:
# from pyscf import tools
# tools.fcidump.from_scf(mf, 'h20.fcidump')

In [46]:
print("mf.e_tot:", mf.e_tot)                  # total HF energy (electronic + nuclear)
print("mol.energy_nuc():", mol.energy_nuc())  # nuclear repulsion
print("electronic HF:", mf.e_tot - mol.energy_nuc())

mf.e_tot: -3.020144532388091
mol.energy_nuc(): 6.576916764291429
electronic HF: -9.59706129667952


In [47]:
import numpy as np
from qiskit.quantum_info import SparsePauliOp, Statevector


def pauli_label(n_qubits, ops):
    """
    Build Qiskit Pauli label.

    ops maps logical qubit index q -> "X", "Y", or "Z".
    Qiskit labels are big-endian: leftmost char is qubit n_qubits-1.
    """
    chars = ["I"] * n_qubits
    for q, p in ops.items():
        chars[n_qubits - 1 - q] = p
    return "".join(chars)


def jw_number(n_qubits, q):
    """
    n_q = (I - Z_q)/2
    """
    I = "I" * n_qubits
    Zq = pauli_label(n_qubits, {q: "Z"})
    return SparsePauliOp.from_list([
        (I, 0.5),
        (Zq, -0.5),
    ])


def jw_hopping(n_qubits, p, q):
    """
    c_p^dagger c_q + c_q^dagger c_p

    Jordan-Wigner:
        1/2 * (X_p Z_{p+1}...Z_{q-1} X_q
             + Y_p Z_{p+1}...Z_{q-1} Y_q)
    for p < q.
    """
    if p == q:
        raise ValueError("p and q must be different")

    if p > q:
        p, q = q, p

    z_string = {r: "Z" for r in range(p + 1, q)}

    x_ops = dict(z_string)
    x_ops[p] = "X"
    x_ops[q] = "X"

    y_ops = dict(z_string)
    y_ops[p] = "Y"
    y_ops[q] = "Y"

    return SparsePauliOp.from_list([
        (pauli_label(n_qubits, x_ops), 0.5),
        (pauli_label(n_qubits, y_ops), 0.5),
    ])


def build_hubbard_sparsepauli(norb, t=1.0, U=4.0):
    n_qubits = 2 * norb

    H = SparsePauliOp.from_list([
        ("I" * n_qubits, 0.0)
    ])

    # spin-up hopping: qubits 0 ... norb-1
    for i in range(norb - 1):
        H += -t * jw_hopping(n_qubits, i, i + 1)

    # spin-down hopping: qubits norb ... 2*norb-1
    for i in range(norb - 1):
        H += -t * jw_hopping(n_qubits, i + norb, i + norb + 1)

    # on-site Hubbard interaction U n_up n_down
    for i in range(norb):
        n_up = jw_number(n_qubits, i)
        n_dn = jw_number(n_qubits, i + norb)
        H += U * n_up.compose(n_dn)

    return H.simplify()

In [48]:
norb = mol.nao_nr()
n_qubits = 2 * norb

H_hubbard = build_hubbard_sparsepauli(
    norb,
    t=1.0,
    U=4.0,
)

sv = Statevector.from_instruction(qc_lucj_raw)

E_qiskit = np.real(sv.expectation_value(H_hubbard))

print("Qiskit exact Hubbard energy:", E_qiskit)

Qiskit exact Hubbard energy: 11.996208765330273
